In [ ]:
pip list

Package                   Version
------------------------- -----------
altair                    5.5.0
anyio                     4.11.0
argon2-cffi               25.1.0
argon2-cffi-bindings      25.1.0
arrow                     1.4.0
asttokens                 3.0.0
async-lru                 2.0.5
attrs                     25.4.0
babel                     2.17.0
beautifulsoup4            4.14.2
bleach                    6.3.0
blinker                   1.9.0
cachetools                6.2.2
certifi                   2025.11.12
cffi                      2.0.0
charset-normalizer        3.4.4
click                     8.3.0
colorama                  0.4.6
comm                      0.2.3
contourpy                 1.3.3
cycler                    0.12.1
debugpy                   1.8.17
decorator                 5.2.1
defusedxml                0.7.1
executing                 2.2.1
fastjsonschema            2.21.2
fonttools                 4.60.1
fqdn                      1.5.1
gitdb            

In [1]:
import os
import pandas as pd
import json
import streamlit as st
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
ag_data_path =r'C:/Users/VICKY/Desktop/Guvi/projects/Project 1/new project/phonepe_dataanalysis/data/aggregated/transaction/country/india/state/'
ag_file_path= [os.path.join(root, filename) for root, dirs, files in os.walk(ag_data_path) for filename in files if filename.endswith('.json')] 
agg_state_list=os.listdir(ag_data_path)

ag_trans_clm={'State':[], 'Year':[],'Quarter':[],'Transaction_type':[], 'Transaction_count':[], 'Transaction_amount':[]}

for i in agg_state_list:
    p_i=ag_data_path+i+"/"
    agg_yr=os.listdir(p_i)
    for j in agg_yr:
        p_j=p_i+j+"/"
        agg_yr_list=os.listdir(p_j)
        for k in agg_yr_list:
            p_k=p_j+k
            Data=open(p_k,'r')
            D=json.load(Data)
            for z in D['data']['transactionData']:
              Name=z['name']
              count=z['paymentInstruments'][0]['count']
              amount=z['paymentInstruments'][0]['amount']
              ag_trans_clm['Transaction_type'].append(Name)
              ag_trans_clm['Transaction_count'].append(count)
              ag_trans_clm['Transaction_amount'].append(amount)
              ag_trans_clm['State'].append(i)
              ag_trans_clm['Year'].append(j)
              ag_trans_clm['Quarter'].append(int(k.strip('.json')))
#Succesfully created a dataframe
agg_Trans=pd.DataFrame(ag_trans_clm)

In [ ]:
agg_Trans

,State,Year,Quarter,Transaction_type,Transaction_count,Transaction_amount
0,andaman-&-nicobar-islands,2018,1,Recharge & bill payments,4200,1.845307e+06
1,andaman-&-nicobar-islands,2018,1,Peer-to-peer payments,1871,1.213866e+07
2,andaman-&-nicobar-islands,2018,1,Merchant payments,298,4.525072e+05
3,andaman-&-nicobar-islands,2018,1,Financial Services,33,1.060142e+04
4,andaman-&-nicobar-islands,2018,1,Others,256,1.846899e+05
...,...,...,...,...,...,...
5029,west-bengal,2024,4,Merchant payments,655100809,3.892862e+11
5030,west-bengal,2024,4,Peer-to-peer payments,493217788,1.361927e+12
5031,west-bengal,2024,4,Recharge & bill payments,76043195,5.753406e+10
5032,west-bengal,2024,4,Financial Services,2352084,8.472965e+08


In [3]:
import pyodbc

In [4]:
import pyodbc

conn = pyodbc.connect(
    r"DRIVER={ODBC Driver 17 for SQL Server};"
    r"SERVER=VIGNESH\SQLEXPRESS;"
    r"DATABASE=phonepe;"
    r"Trusted_Connection=yes;"
)

cursor = conn.cursor()
print("Connected successfully!")


Connected successfully!


In [5]:
# 1) Inspect your DataFrame
print(agg_Trans.shape)
print(agg_Trans.columns.tolist())       # EXACT column names (case sensitive for attribute access)
print(agg_Trans.head())

# 2) Make sure columns match the SQL table columns
# expected: ['year','quarter','state','transaction_type','transaction_count','transaction_amount']


(5034, 6)
['State', 'Year', 'Quarter', 'Transaction_type', 'Transaction_count', 'Transaction_amount']
                       State  Year  Quarter          Transaction_type  \
0  andaman-&-nicobar-islands  2018        1  Recharge & bill payments   
1  andaman-&-nicobar-islands  2018        1     Peer-to-peer payments   
2  andaman-&-nicobar-islands  2018        1         Merchant payments   
3  andaman-&-nicobar-islands  2018        1        Financial Services   
4  andaman-&-nicobar-islands  2018        1                    Others   

   Transaction_count  Transaction_amount  
0               4200        1.845307e+06  
1               1871        1.213866e+07  
2                298        4.525072e+05  
3                 33        1.060142e+04  
4                256        1.846899e+05  


In [6]:
import pyodbc
import pandas as pd
import numpy as np

conn = pyodbc.connect(
    r"DRIVER={ODBC Driver 17 for SQL Server};"
    r"SERVER=VIGNESH\SQLEXPRESS;"
    r"DATABASE=phonepe;"
    r"Trusted_Connection=yes;"
)
cursor = conn.cursor()

insert_sql = """
INSERT INTO dbo.aggregated_transaction
(Year, Quarter, State, Transaction_type, Transaction_count, Transaction_amount)
VALUES (?, ?, ?, ?, ?, ?)
"""

data_iter = [
    (int(r.Year), int(r.Quarter), r.State, r.Transaction_type, int(r.Transaction_count), float(r.Transaction_amount))
    for r in agg_Trans.itertuples(index=False)
]

cursor.fast_executemany = True   # speeds up executemany
cursor.executemany(insert_sql, data_iter)
conn.commit()

print("Inserted", cursor.rowcount, "rows (last batch rowcount).")

cursor.close()
conn.close()


Inserted -1 rows (last batch rowcount).


In [7]:
ag_ins_datapath=r'C:/Users/VICKY/Desktop/Guvi/projects/Project 1/new project/phonepe_dataanalysis/data/aggregated/insurance/country/india/state/'
ag_ins_state_list=os.listdir(ag_ins_datapath)

ag_ins_clm={'State':[], 'Year':[],'Quarter':[],'insurance_type':[], 'insurance_count':[], 'insurance_amount':[]}

for i in ag_ins_state_list:
    p_i=ag_ins_datapath+i+"/"
    ag_ins_yr=os.listdir(p_i)
    for j in ag_ins_yr:
        p_j=p_i+j+"/"
        ag_ins_yr_list=os.listdir(p_j)
        for k in ag_ins_yr_list:
            p_k=p_j+k
            Data=open(p_k,'r')
            D=json.load(Data)
            for z in D['data']['transactionData']:
              Name=z['name']
              count=z['paymentInstruments'][0]['count']
              amount=z['paymentInstruments'][0]['amount']
              ag_ins_clm['insurance_type'].append(Name)
              ag_ins_clm['insurance_count'].append(count)
              ag_ins_clm['insurance_amount'].append(amount)
              ag_ins_clm['State'].append(i)
              ag_ins_clm['Year'].append(j)
              ag_ins_clm['Quarter'].append(int(k.strip('.json')))
#Succesfully created a dataframe
agg_ins=pd.DataFrame(ag_ins_clm)

In [8]:
import pyodbc
import pandas as pd
import numpy as np

conn = pyodbc.connect(
    r"DRIVER={ODBC Driver 17 for SQL Server};"
    r"SERVER=VIGNESH\SQLEXPRESS;"
    r"DATABASE=phonepe;"
    r"Trusted_Connection=yes;"
)
cursor = conn.cursor()

insert_sql = """
INSERT INTO dbo.aggregated_insurance
(Year, Quarter, State, insurance_type, insurance_count, insurance_amount)
VALUES (?, ?, ?, ?, ?, ?)
"""

data_iter = [
    (int(r.Year), int(r.Quarter), r.State, r.insurance_type, int(r.insurance_count), float(r.insurance_amount))
    for r in agg_ins.itertuples(index=False)
]

cursor.fast_executemany = True   # speeds up executemany
cursor.executemany(insert_sql, data_iter)
conn.commit()

print("Inserted", cursor.rowcount, "rows (last batch rowcount).")

cursor.close()
conn.close()


Inserted -1 rows (last batch rowcount).


In [9]:
agg_ins.tail()

,State,Year,Quarter,insurance_type,insurance_count,insurance_amount
677,west-bengal,2023,4,Insurance,72712,100365562.0
678,west-bengal,2024,1,Insurance,79576,104987909.0
679,west-bengal,2024,2,Insurance,67048,89476633.0
680,west-bengal,2024,3,Insurance,77158,107451766.0
681,west-bengal,2024,4,Insurance,91719,120602777.0


In [10]:
import os
import pandas as pd
user_datapath=r'C:/Users/VICKY/Desktop/Guvi/projects/Project 1/new project/phonepe_dataanalysis/data/aggregated/user/country/india/state/'
ag_user=os.listdir(user_datapath)

ag_user_clm={'State':[], 'Year':[],'Quarter':[],'user_brand':[], 'user_count':[], 'user_percentage':[]}

for i in ag_user:
    p_i=user_datapath+i+"/"
    ag_user_yr=os.listdir(p_i)
    for j in ag_user_yr:
        p_j=p_i+j+"/"
        ag_user_yr_list=os.listdir(p_j)
        for k in ag_user_yr_list:
            p_k=p_j+k
            print(p_k)
            Data=open(p_k,'r')
            D=json.load(Data)
            print(D)
            
            users_by_device = D['data'].get('usersByDevice')
            if not users_by_device:
                continue
            for device in users_by_device:
               
                brand = device.get('brand') or device.get('Brand') or 'Unknown'
                count = device.get('count', device.get('Users_Count', 0)) 
                percentage = device.get('percentage', 0.0) 
                ag_user_clm['user_brand'].append(brand)
                ag_user_clm['user_count'].append(count)
                ag_user_clm['user_percentage'].append(percentage)
                ag_user_clm['State'].append(i)
                ag_user_clm['Year'].append(j)
                ag_user_clm['Quarter'].append(int(k.strip('.json')))
#Succesfully created a dataframe
agg_user=pd.DataFrame(ag_user_clm)

C:/Users/VICKY/Desktop/Guvi/projects/Project 1/new project/phonepe_dataanalysis/data/aggregated/user/country/india/state/andaman-&-nicobar-islands/2018/1.json
{'success': True, 'code': 'SUCCESS', 'data': {'aggregated': {'registeredUsers': 6740, 'appOpens': 0}, 'usersByDevice': [{'brand': 'Xiaomi', 'count': 1665, 'percentage': 0.2470326409495549}, {'brand': 'Samsung', 'count': 1445, 'percentage': 0.21439169139465875}, {'brand': 'Vivo', 'count': 982, 'percentage': 0.1456973293768546}, {'brand': 'Oppo', 'count': 501, 'percentage': 0.07433234421364986}, {'brand': 'OnePlus', 'count': 332, 'percentage': 0.04925816023738872}, {'brand': 'Realme', 'count': 316, 'percentage': 0.04688427299703264}, {'brand': 'Apple', 'count': 229, 'percentage': 0.03397626112759644}, {'brand': 'Motorola', 'count': 226, 'percentage': 0.03353115727002967}, {'brand': 'Lenovo', 'count': 202, 'percentage': 0.02997032640949555}, {'brand': 'Huawei', 'count': 158, 'percentage': 0.02344213649851632}, {'brand': 'Others', 'c

KeyboardInterrupt: 

In [ ]:
agg_user

,State,Year,Quarter,user_brand,user_count,user_percentage
0,andaman-&-nicobar-islands,2018,1,Xiaomi,1665,0.247033
1,andaman-&-nicobar-islands,2018,1,Samsung,1445,0.214392
2,andaman-&-nicobar-islands,2018,1,Vivo,982,0.145697
3,andaman-&-nicobar-islands,2018,1,Oppo,501,0.074332
4,andaman-&-nicobar-islands,2018,1,OnePlus,332,0.049258
...,...,...,...,...,...,...
6727,west-bengal,2022,1,Lenovo,330017,0.015056
6728,west-bengal,2022,1,Infinix,284678,0.012987
6729,west-bengal,2022,1,Asus,280347,0.012790
6730,west-bengal,2022,1,Apple,277752,0.012671


In [ ]:
conn = pyodbc.connect(
    r"DRIVER={ODBC Driver 17 for SQL Server};"
    r"SERVER=VIGNESH\SQLEXPRESS;"
    r"DATABASE=phonepe;"
    r"Trusted_Connection=yes;"
)
cursor = conn.cursor()

In [ ]:
import pyodbc
import pandas as pd
import numpy as np

conn = pyodbc.connect(
    r"DRIVER={ODBC Driver 17 for SQL Server};"
    r"SERVER=VIGNESH\SQLEXPRESS;"
    r"DATABASE=phonepe;"
    r"Trusted_Connection=yes;"
)
cursor = conn.cursor()

insert_sql = """
INSERT INTO aggregated_user
(Year, Quarter, State, user_brand, user_count, user_percentage)
VALUES (?, ?, ?, ?, ?, ?)
"""

data_iter = [
    (int(r.Year), int(r.Quarter), r.State, r.user_brand, int(r.user_count), float(r.user_percentage))
    for r in agg_user.itertuples(index=False)
]

cursor.fast_executemany = True   # speeds up executemany
cursor.executemany(insert_sql,data_iter)
conn.commit()

print("Inserted", cursor.rowcount, "rows (last batch rowcount).")

cursor.close()
conn.close()


Inserted -1 rows (last batch rowcount).


In [ ]:
agg_user

,State,Year,Quarter,user_brand,user_count,user_percentage
0,andaman-&-nicobar-islands,2018,1,Xiaomi,1665,0.247033
1,andaman-&-nicobar-islands,2018,1,Samsung,1445,0.214392
2,andaman-&-nicobar-islands,2018,1,Vivo,982,0.145697
3,andaman-&-nicobar-islands,2018,1,Oppo,501,0.074332
4,andaman-&-nicobar-islands,2018,1,OnePlus,332,0.049258
...,...,...,...,...,...,...
6727,west-bengal,2022,1,Lenovo,330017,0.015056
6728,west-bengal,2022,1,Infinix,284678,0.012987
6729,west-bengal,2022,1,Asus,280347,0.012790
6730,west-bengal,2022,1,Apple,277752,0.012671


In [11]:
m_trans_datapath=r'C:/Users/VICKY/Desktop/Guvi/projects/Project 1/new project/phonepe_dataanalysis/data/map/transaction//hover/country/india/state/'
m_trans_state_list=os.listdir(m_trans_datapath)

m_trans_clm={'State':[], 'Year':[],'Quarter':[],'m_transaction_type':[], 'm_transaction_count':[], 'm_transaction_amount':[]}

for i in m_trans_state_list:
    p_i=m_trans_datapath+i+"/"
    m_yr=os.listdir(p_i)
    for j in m_yr:
        p_j=p_i+j+"/"
        m_yr_list=os.listdir(p_j)
        for k in m_yr_list:
            p_k=p_j+k
            Data=open(p_k,'r')
            D=json.load(Data)
            for z in D['data']['hoverDataList']:
              Name=z['name']
              count=z['metric'][0]['count']
              amount=z['metric'][0]['amount']
              m_trans_clm['m_transaction_type'].append(Name)
              m_trans_clm['m_transaction_count'].append(count)
              m_trans_clm['m_transaction_amount'].append(amount)
              m_trans_clm['State'].append(i)
              m_trans_clm['Year'].append(j)
              m_trans_clm['Quarter'].append(int(k.strip('.json')))
#Succesfully created a dataframe
m_trans=pd.DataFrame(m_trans_clm)

In [ ]:
import pyodbc
import pandas as pd
import numpy as np

conn = pyodbc.connect(
    r"DRIVER={ODBC Driver 17 for SQL Server};"
    r"SERVER=VIGNESH\SQLEXPRESS;"
    r"DATABASE=phonepe;"
    r"Trusted_Connection=yes;"
)
cursor = conn.cursor()

insert_sql = """
INSERT INTO map_transaction
(Year, Quarter, State, m_transaction_type, m_transaction_count, m_transaction_amount)
VALUES (?, ?, ?, ?, ?, ?)
"""

data_iter = [
    (int(r.Year), int(r.Quarter), r.State, r.m_transaction_type, int(r.m_transaction_count), float(r.m_transaction_amount))
    for r in m_trans.itertuples(index=False)
]

cursor.fast_executemany = True   # speeds up executemany
cursor.executemany(insert_sql,data_iter)
conn.commit()

print("Inserted", cursor.rowcount, "rows (last batch rowcount).")

cursor.close()
conn.close()


Inserted -1 rows (last batch rowcount).


In [ ]:
m_trans

,State,Year,Quarter,m_transaction_type,m_transaction_count,m_transaction_amount
0,andaman-&-nicobar-islands,2018,1,north and middle andaman district,442,9.316631e+05
1,andaman-&-nicobar-islands,2018,1,south andaman district,5688,1.256025e+07
2,andaman-&-nicobar-islands,2018,1,nicobars district,528,1.139849e+06
3,andaman-&-nicobar-islands,2018,2,north and middle andaman district,825,1.317863e+06
4,andaman-&-nicobar-islands,2018,2,south andaman district,9395,2.394824e+07
...,...,...,...,...,...,...
20599,west-bengal,2024,4,alipurduar district,15875637,2.099251e+10
20600,west-bengal,2024,4,paschim bardhaman district,56616799,6.968735e+10
20601,west-bengal,2024,4,nadia district,65274337,1.079320e+11
20602,west-bengal,2024,4,birbhum district,36905213,5.778701e+10


In [12]:
m_user_datapath = r'C:/Users/VICKY/Desktop/Guvi/projects/Project 1/new project/phonepe_dataanalysis/data/map/users/hover/country/india/state/'

In [14]:
import os
import json
import pandas as pd

# Define the path
m_user_datapath = r'C:/Users/VICKY/Desktop/Guvi/projects/Project 1/new project/phonepe_dataanalysis/data/map/user/hover/country/india/state/'

m_clm_user = {'State': [],'Year': [], 'Quarter': [],'District': [],'m_registered_Users': [],'m_app_Opens': []}    
   

for state in os.listdir(m_user_datapath):
    state_path = os.path.join(m_user_datapath, state)
    for year in os.listdir(state_path):
        year_path = os.path.join(state_path, year)
        for quarter_file in os.listdir(year_path):
            if quarter_file.endswith('.json'):
                file_path = os.path.join(year_path, quarter_file)  
                with open(file_path, 'r') as file:
                    json_data = json.load(file)
                quarter = int(quarter_file.replace('.json', ''))
                hover_data = json_data['data']['hoverData']
                for district, metrics in hover_data.items():
                    m_clm_user['State'].append(state)
                    m_clm_user['Year'].append(year)
                    m_clm_user['Quarter'].append(quarter)
                    m_clm_user['District'].append(district)
                    m_clm_user['m_registered_Users'].append(metrics['registeredUsers'])
                    m_clm_user['m_app_Opens'].append(metrics['appOpens'])

m_user = pd.DataFrame(m_clm_user)



In [15]:
import pyodbc
import pandas as pd
import numpy as np

conn = pyodbc.connect(
    r"DRIVER={ODBC Driver 17 for SQL Server};"
    r"SERVER=VIGNESH\SQLEXPRESS;"
    r"DATABASE=phonepe;"
    r"Trusted_Connection=yes;"
)
cursor = conn.cursor()

insert_sql = """
INSERT INTO map_user
(Year, Quarter, State, District, m_registered_Users, m_app_Opens)
VALUES (?, ?, ?, ?, ?, ?)
"""

data_iter = [
    (int(r.Year), int(r.Quarter), r.State, r.District, int(r.m_registered_Users), int(r.m_app_Opens))
    for r in m_user.itertuples(index=False)
]

cursor.fast_executemany = True   # speeds up executemany
cursor.executemany(insert_sql,data_iter)
conn.commit()

print("Inserted", cursor.rowcount, "rows (last batch rowcount).")

cursor.close()
conn.close()


Inserted -1 rows (last batch rowcount).


In [16]:
m_user

,State,Year,Quarter,District,m_registered_Users,m_app_Opens
0,andaman-&-nicobar-islands,2018,1,north and middle andaman district,632,0
1,andaman-&-nicobar-islands,2018,1,south andaman district,5846,0
2,andaman-&-nicobar-islands,2018,1,nicobars district,262,0
3,andaman-&-nicobar-islands,2018,2,north and middle andaman district,911,0
4,andaman-&-nicobar-islands,2018,2,south andaman district,8143,0
...,...,...,...,...,...,...
20603,west-bengal,2024,4,alipurduar district,475688,31842355
20604,west-bengal,2024,4,paschim bardhaman district,1468252,80543469
20605,west-bengal,2024,4,nadia district,1861738,98740305
20606,west-bengal,2024,4,birbhum district,1114220,73465525


In [17]:
m_ins_datapath=r'C:/Users/VICKY/Desktop/Guvi/projects/Project 1/new project/phonepe_dataanalysis/data/map/insurance/hover/country/india/state/'
m_ins_state_list=os.listdir(m_ins_datapath)

m_ins_clm={'State':[], 'Year':[],'Quarter':[],'m_insurance_type':[], 'm_insurance_count':[], 'm_insurance_amount':[]}

for i in m_ins_state_list:
    p_i=m_ins_datapath+i+"/"
    m_ins_yr=os.listdir(p_i)
    for j in m_ins_yr:
        p_j=p_i+j+"/"
        m_ins_yr_list=os.listdir(p_j)
        for k in m_ins_yr_list:
            p_k=p_j+k
            Data=open(p_k,'r')
            D=json.load(Data)
            for z in D['data']['hoverDataList']:
                Name = z['name']
                count = z['metric'][0]['count']
                amount = z['metric'][0]['amount']
                m_ins_clm['m_insurance_type'].append(Name)
                m_ins_clm['m_insurance_count'].append(count)
                m_ins_clm['m_insurance_amount'].append(amount)
                m_ins_clm['State'].append(i)
                m_ins_clm['Year'].append(j)
                m_ins_clm['Quarter'].append(int(k.strip('.json')))
    
#Succesfully created a dataframe
m_ins=pd.DataFrame(m_ins_clm)

In [18]:
import pyodbc
import pandas as pd
import numpy as np

conn = pyodbc.connect(
    r"DRIVER={ODBC Driver 17 for SQL Server};"
    r"SERVER=VIGNESH\SQLEXPRESS;"
    r"DATABASE=phonepe;"
    r"Trusted_Connection=yes;"
)
cursor = conn.cursor()

insert_sql = """
INSERT INTO map_insurance
(Year, Quarter, State, m_insurance_type, m_insurance_count, m_insurance_amount)
VALUES (?, ?, ?, ?, ?, ?)
"""

data_iter = [
    (int(r.Year), int(r.Quarter), r.State, r.m_insurance_type, int(r.m_insurance_count), int(r.m_insurance_amount))
    for r in m_ins .itertuples(index=False)
]

cursor.fast_executemany = True   # speeds up executemany
cursor.executemany(insert_sql,data_iter)
conn.commit()

print("Inserted", cursor.rowcount, "rows (last batch rowcount).")

cursor.close()
conn.close()


Inserted -1 rows (last batch rowcount).


In [19]:
m_ins

,State,Year,Quarter,m_insurance_type,m_insurance_count,m_insurance_amount
0,andaman-&-nicobar-islands,2020,2,south andaman district,3,795.0
1,andaman-&-nicobar-islands,2020,2,nicobars district,3,565.0
2,andaman-&-nicobar-islands,2020,3,north and middle andaman district,1,281.0
3,andaman-&-nicobar-islands,2020,3,south andaman district,35,13651.0
4,andaman-&-nicobar-islands,2020,3,nicobars district,5,1448.0
...,...,...,...,...,...,...
13871,west-bengal,2024,4,alipurduar district,1023,1613143.0
13872,west-bengal,2024,4,paschim bardhaman district,4945,7005851.0
13873,west-bengal,2024,4,nadia district,3807,5031294.0
13874,west-bengal,2024,4,birbhum district,1818,2423290.0


In [20]:

top_trans_path = r'C:/Users/VICKY/Desktop/Guvi/projects/Project 1/new project/phonepe_dataanalysis/data/top/transaction/country/india/'

# Dictionaries to collect data
state_cols = {
    'Year': [],
    'Quarter': [],
    'State': [],
    'Metric_Type': [],
    'Transaction_Count': [],
    'Transaction_Amount': []
}

district_cols = {
    'Year': [],
    'Quarter': [],
    'District': [],
    'Metric_Type': [],
    'Transaction_Count': [],
    'Transaction_Amount': []
}

pincode_cols = {
    'Year': [],
    'Quarter': [],
    'Pincode': [],
    'Metric_Type': [],
    'Transaction_Count': [],
    'Transaction_Amount': []
}

# Loop through year folders
year_list = os.listdir(top_trans_path)

for year in year_list:
    year_path = os.path.join(top_trans_path, year)
    if not os.path.isdir(year_path):
        continue

    # Loop through quarter json files inside the year folder
    quarter_files = os.listdir(year_path)

    for q_file in quarter_files:
        if not q_file.endswith('.json'):
            continue

        q_path = os.path.join(year_path, q_file)
        quarter = int(q_file.strip('.json'))  # '1.json' -> 1

        with open(q_path, 'r', encoding='utf-8') as f:
            D = json.load(f)

        # -------------------- STATES --------------------
        states_list = D['data'].get('states')
        if states_list:
            for s in states_list:
                state_cols['Year'].append(year)
                state_cols['Quarter'].append(quarter)
                state_cols['State'].append(s['entityName'])
                state_cols['Metric_Type'].append(s['metric']['type'])
                state_cols['Transaction_Count'].append(s['metric']['count'])
                state_cols['Transaction_Amount'].append(s['metric']['amount'])

        # -------------------- DISTRICTS --------------------
        districts_list = D['data'].get('districts')
        if districts_list:
            for d in districts_list:
                district_cols['Year'].append(year)
                district_cols['Quarter'].append(quarter)
                district_cols['District'].append(d['entityName'])
                district_cols['Metric_Type'].append(d['metric']['type'])
                district_cols['Transaction_Count'].append(d['metric']['count'])
                district_cols['Transaction_Amount'].append(d['metric']['amount'])

        # -------------------- PINCODES --------------------
        pincodes_list = D['data'].get('pincodes')
        if pincodes_list:
            for p in pincodes_list:
                pincode_cols['Year'].append(year)
                pincode_cols['Quarter'].append(quarter)
                pincode_cols['Pincode'].append(p['entityName'])
                pincode_cols['Metric_Type'].append(p['metric']['type'])
                pincode_cols['Transaction_Count'].append(p['metric']['count'])
                pincode_cols['Transaction_Amount'].append(p['metric']['amount'])

# Create DataFrames
top_trans_state_df = pd.DataFrame(state_cols)
top_trans_district_df = pd.DataFrame(district_cols)
top_trans_pincode_df = pd.DataFrame(pincode_cols)



In [21]:
import pyodbc
import pandas as pd

conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=VIGNESH\\SQLEXPRESS;"
    "DATABASE=phonepe;"
    "Trusted_Connection=yes;"
)

cursor = conn.cursor()
print("Connected to SQL Server!")

from sqlalchemy import create_engine

conn_str = (
    "mssql+pyodbc://VIGNESH\\SQLEXPRESS/phonepe?"
    "driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
)

engine = create_engine(conn_str)

top_trans_state_df.to_sql(
    name="top_state_transaction",
    con=engine,
    if_exists="append",
    index=False
)
print("State data inserted!")

top_trans_district_df.to_sql(
    name="top_district_transaction",
    con=engine,
    if_exists="append",
    index=False
)
print("District data inserted!")

top_trans_pincode_df.to_sql(
    name="top_pincode_transaction",
    con=engine,
    if_exists="append",
    index=False
)
print("pincode data inserted!")


Connected to SQL Server!
State data inserted!
District data inserted!
pincode data inserted!


In [23]:
import os
import json
import pandas as pd

top_user_path = r'C:/Users/VICKY/Desktop/Guvi/projects/Project 1/new project/phonepe_dataanalysis/data/top/user/country/india/'

# Dictionaries to collect data
state_cols = {
    'Year': [],
    'Quarter': [],
    'State': [],
    'registeredUsers': []
}

district_cols = {
    'Year': [],
    'Quarter': [],
    'District': [],
    'registeredUsers': []
}

pincode_cols = {
    'Year': [],
    'Quarter': [],
    'Pincode': [],
    'registeredUsers': []
}

# Loop through year folders
year_list = os.listdir(top_user_path)

for year in year_list:
    year_path = os.path.join(top_user_path, year)
    if not os.path.isdir(year_path):
        continue

    # Loop through quarter json files inside the year folder
    quarter_files = os.listdir(year_path)

    for q_file in quarter_files:
        if not q_file.endswith('.json'):
            continue

        q_path = os.path.join(year_path, q_file)
        quarter = int(q_file.replace('.json', ''))

        with open(q_path, 'r', encoding='utf-8') as f:
            D = json.load(f)

        # -------------------- STATES --------------------
        states_list = D['data'].get('states', [])
        for s in states_list:
            state_cols['Year'].append(int(year))
            state_cols['Quarter'].append(quarter)
            state_cols['State'].append(s["name"])
            state_cols['registeredUsers'].append(s["registeredUsers"])

        # -------------------- DISTRICTS --------------------
        districts_list = D['data'].get('districts', [])
        for d in districts_list:
            district_cols['Year'].append(int(year))
            district_cols['Quarter'].append(quarter)
            district_cols['District'].append(d["name"])
            district_cols['registeredUsers'].append(d["registeredUsers"])

        # -------------------- PINCODES --------------------
        pincodes_list = D['data'].get('pincodes', [])
        for p in pincodes_list:
            pincode_cols['Year'].append(int(year))
            pincode_cols['Quarter'].append(quarter)
            pincode_cols['Pincode'].append(p["name"])
            pincode_cols['registeredUsers'].append(p["registeredUsers"])

# Create DataFrames
top_user_state_df = pd.DataFrame(state_cols)
top_user_district_df = pd.DataFrame(district_cols)
top_user_pincode_df = pd.DataFrame(pincode_cols)

print(top_user_state_df.head())
print(top_user_district_df.head())
print(top_user_pincode_df.head())


   Year  Quarter           State  registeredUsers
0  2018        1     maharashtra          6106994
1  2018        1   uttar pradesh          4694250
2  2018        1       karnataka          3717763
3  2018        1  andhra pradesh          3336450
4  2018        1       telangana          3315560
   Year  Quarter         District  registeredUsers
0  2018        1  bengaluru urban          1922368
1  2018        1             pune          1211643
2  2018        1           jaipur           900773
3  2018        1  mumbai suburban           719300
4  2018        1        hyderabad           655175
   Year  Quarter Pincode  registeredUsers
0  2018        1  201301           114625
1  2018        1  500072           105012
2  2018        1  560068            98487
3  2018        1  110059            95496
4  2018        1  110092            83600


In [24]:
import sqlalchemy as sa
import pyodbc
import pandas as pd

conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=VIGNESH\\SQLEXPRESS;"
    "DATABASE=phonepe;"
    "Trusted_Connection=yes;"
)

cursor = conn.cursor()
print("Connected to SQL Server!")
# Connection string for MSSQL
conn_str = (
    "mssql+pyodbc://VIGNESH\\SQLEXPRESS/phonepe?"
    "driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
)

engine = create_engine(conn_str)

# Insert state dataframe
top_user_state_df.to_sql(
    "top_user_state",
    con=engine,
    if_exists="append",
    index=False
)

# Insert district dataframe
top_user_district_df.to_sql(
    "top_user_district",
    con=engine,
    if_exists="append",
    index=False
)

# Insert pincode dataframe
top_user_pincode_df.to_sql(
    "top_user_pincode",
    con=engine,
    if_exists="append",
    index=False
)

print("All 3 dataframes inserted successfully!")


Connected to SQL Server!


All 3 dataframes inserted successfully!


In [26]:

top_ins__path = r'C:/Users/VICKY/Desktop/Guvi/projects/Project 1/new project/phonepe_dataanalysis/data/top/insurance/country/india/'

# Dictionaries to collect data
top_ins_state_cols = {
    'Year': [],
    'Quarter': [],
    'State': [],
    'Metric_Type': [],
    'Transaction_Count': [],
    'Transaction_Amount': []
}

top_ins_district_cols = {
    'Year': [],
    'Quarter': [],
    'District': [],
    'Metric_Type': [],
    'Transaction_Count': [],
    'Transaction_Amount': []
}

top_ins_pincode_cols = {
    'Year': [],
    'Quarter': [],
    'Pincode': [],
    'Metric_Type': [],
    'Transaction_Count': [],
    'Transaction_Amount': []
}

# Loop through year folders
year_list = os.listdir(top_ins__path)

for year in year_list:
    year_path = os.path.join(top_ins__path, year)
    if not os.path.isdir(year_path):
        continue

    # Loop through quarter json files inside the year folder
    quarter_files = os.listdir(year_path)

    for q_file in quarter_files:
        if not q_file.endswith('.json'):
            continue

        q_path = os.path.join(year_path, q_file)
        quarter = int(q_file.strip('.json'))  # '1.json' -> 1

        with open(q_path, 'r', encoding='utf-8') as f:
            D = json.load(f)

        # -------------------- STATES --------------------
        states_list = D['data'].get('states')
        if states_list:
            for s in states_list:
                top_ins_state_cols['Year'].append(year)
                top_ins_state_cols['Quarter'].append(quarter)
                top_ins_state_cols['State'].append(s['entityName'])
                top_ins_state_cols['Metric_Type'].append(s['metric']['type'])
                top_ins_state_cols['Transaction_Count'].append(s['metric']['count'])
                top_ins_state_cols['Transaction_Amount'].append(s['metric']['amount'])

        # -------------------- DISTRICTS --------------------
        districts_list = D['data'].get('districts')
        if districts_list:
            for d in districts_list:
                top_ins_district_cols['Year'].append(year)
                top_ins_district_cols['Quarter'].append(quarter)
                top_ins_district_cols['District'].append(d['entityName'])
                top_ins_district_cols['Metric_Type'].append(d['metric']['type'])
                top_ins_district_cols['Transaction_Count'].append(d['metric']['count'])
                top_ins_district_cols['Transaction_Amount'].append(d['metric']['amount'])

        # -------------------- PINCODES --------------------
        pincodes_list = D['data'].get('pincodes')
        if pincodes_list:
            for p in pincodes_list:
                top_ins_pincode_cols['Year'].append(year)
                top_ins_pincode_cols['Quarter'].append(quarter)
                top_ins_pincode_cols['Pincode'].append(p['entityName'])
                top_ins_pincode_cols['Metric_Type'].append(p['metric']['type'])
                top_ins_pincode_cols['Transaction_Count'].append(p['metric']['count'])
                top_ins_pincode_cols['Transaction_Amount'].append(p['metric']['amount'])

# Create DataFrames
top_ins__state_df = pd.DataFrame(top_ins_state_cols)
top_ins__district_df = pd.DataFrame(top_ins_district_cols)
top_ins__pincode_df = pd.DataFrame(top_ins_pincode_cols)



In [27]:
top_ins__state_df

,Year,Quarter,State,Metric_Type,Transaction_Count,Transaction_Amount
0,2020,2,maharashtra,TOTAL,39836,6879717.0
1,2020,2,karnataka,TOTAL,27358,4794150.0
2,2020,2,andhra pradesh,TOTAL,22104,3982391.0
3,2020,2,telangana,TOTAL,19003,3419453.0
4,2020,2,delhi,TOTAL,11716,1897480.0
...,...,...,...,...,...,...
185,2024,4,kerala,TOTAL,89533,144083113.0
186,2024,4,telangana,TOTAL,78498,124755619.0
187,2024,4,rajasthan,TOTAL,73530,127930986.0
188,2024,4,delhi,TOTAL,67962,94390728.0


In [28]:
import sqlalchemy as sa
import pyodbc
import pandas as pd

conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=VIGNESH\\SQLEXPRESS;"
    "DATABASE=phonepe;"
    "Trusted_Connection=yes;"
)

cursor = conn.cursor()
print("Connected to SQL Server!")
# Connection string for MSSQL
conn_str = (
    "mssql+pyodbc://VIGNESH\\SQLEXPRESS/phonepe?"
    "driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
)

engine= create_engine(conn_str)

# Insert state dataframe
top_ins__state_df.to_sql(
    "top_ins_state",
    con=engine
    ,
    if_exists="append",
    index=False
)

# Insert district dataframe
top_ins__district_df.to_sql(
    "top_ins_district",
    con=engine
    ,
    if_exists="append",
    index=False
)

# Insert pincode dataframe
top_ins__pincode_df.to_sql(
    "top_ins_pincode",
    con=engine
    ,
    if_exists="append",
    index=False
)

print("All 3 dataframes inserted successfully!")


Connected to SQL Server!


All 3 dataframes inserted successfully!
